# Loan Approval Prediction & Credit Risk Analysis
## A Business-Intelligence Data Analytics Project

**Program:** AICTE | IBM SkillsBuild Internship Program 2026 — *Data Analytics with AI: Foundation to Implementation*
**Sponsored by:** BharatCares &nbsp;|&nbsp; **Intern:** Gupta Vaishnavi Sureshbhai &nbsp;|&nbsp; **Date:** 23 September 2026

**Tools used:** Python, Pandas, NumPy, Matplotlib, Scikit-learn — in a single Jupyter notebook (Colab-compatible).

---

I structured this project around the BI flow we covered in the program — going from raw data all the way to actual business recommendations, not just model accuracy. The five levels below are how I organised my thinking:

| Level | What I was trying to answer |
|---|---|
| 1. **KPIs** | What is actually happening in this loan portfolio? |
| 2. **Trends** | Are there patterns across groups or segments? |
| 3. **Drivers** | Which factors actually explain why someone gets approved or rejected? |
| 4. **Risks & Opportunities** | What should the bank be worried about — and what could it do better? |
| 5. **Actions** | Concrete things the bank could actually change |

The prediction model I built at the end is logistic regression — I chose it specifically because you can explain the output, which matters a lot in banking.


## 1. Problem Statement

I picked this dataset because loan decisions feel like a black box — you apply, the bank says yes or no, and you often don't really know why. I wanted to see if I could reverse-engineer what's actually driving those decisions from historical data.

The dataset has 4,269 loan applications with 13 columns: income, CIBIL score, loan amount, loan term, asset values, education, employment status, and the final approval/rejection call. My goals going in:

1. Figure out which factors actually matter for approval (and which ones *don't* — that's often just as interesting).
2. Find which applicant profiles are high-risk and which ones are safe to lend to.
3. Build a simple model that can pre-screen new applications and tell you *why* it made a decision.
4. Turn all of that into actual recommendations — not just charts.

I went in assuming income would be the main driver. Turned out I was wrong about that, which ended up being one of the more interesting findings.


## 2. Setup — imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report)
import os

os.makedirs('outputs', exist_ok=True)

PRIMARY = '#0F62FE'   # IBM blue
ACCENT  = '#FF832B'   # orange
GREEN   = '#198038'
RED     = '#DA1E28'
GRAY    = '#8D8D8D'

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   'white',
    'axes.edgecolor':   '#D0D0D0',
    'axes.grid':        True,
    'grid.alpha':       0.3,
    'grid.linewidth':   0.8,
    'axes.axisbelow':   True,
    'font.size':        11,
    'axes.titlesize':   13,
    'axes.titleweight': 'bold',
    'axes.labelsize':   11,
    'xtick.color':      '#333333',
    'ytick.color':      '#333333',
})
print('Setup complete.')


## 3. Data Collection & Loading

Loading the dataset from `data/loan_approval_dataset.csv` (4,269 rows × 13 columns). The fallback URL loads it from GitHub if running on Colab.


In [ ]:
csv_path = None
for candidate in ['data/loan_approval_dataset.csv', 'loan_approval_dataset.csv']:
    if os.path.exists(candidate):
        csv_path = candidate
        break

if csv_path is not None:
    df = pd.read_csv(csv_path)
    print(f'Loaded from local file: {csv_path}')
else:
    url = ('https://raw.githubusercontent.com/sarahrafiqshaikh/'
           'Loan-Approval-Prediction-Analysis/main/loan_approval_dataset.csv')
    df = pd.read_csv(url)
    print('Loaded from GitHub fallback')

df.head()


Before cleaning anything, I want to get a feel for the shape of the data — types, ranges, whether anything looks off.

In [ ]:
# First look — always do this before anything else
print(df.shape)
df.info()


In [ ]:
# Quick stats on all numeric columns
df.describe()


## 4. Data Cleaning & Quality Checks

Running through the usual checklist — missing values, duplicates, data type issues, value ranges. One thing I've learned is to always check for stray whitespace in column names and string values. It looks clean but breaks every groupby and filter silently.


In [ ]:
# 4.1 Strip whitespace from column names and string columns
# (this tripped me up on an earlier project — 'education' vs ' education' look the same but aren't)
df.columns = df.columns.str.strip()
for c in df.select_dtypes('object').columns:
    df[c] = df[c].str.strip()
print('Column names after cleaning:')
print(list(df.columns))


In [ ]:
# 4.2 Missing values — hoping for none, let's see
missing = df.isnull().sum()
print('Missing values per column:')
print(missing)
print(f'\nTotal missing cells: {missing.sum()}')


In [ ]:
# 4.3 Duplicates
print('Duplicate loan IDs :', df['loan_id'].duplicated().sum())
print('Fully duplicate rows:', df.duplicated().sum())


In [ ]:
# 4.4 Sanity check on ranges
print('CIBIL score range :', df['cibil_score'].min(), '-', df['cibil_score'].max(), '(expected 300-900)')
print('Loan terms present:', sorted(df['loan_term'].unique()))
print('Decision values   :', sorted(df['loan_status'].unique()))

# Negative asset values don't make sense — check for them
asset_cols = ['residential_assets_value', 'commercial_assets_value',
              'luxury_assets_value', 'bank_asset_value']
neg = (df[asset_cols] < 0).sum()
print('\nNegative asset values:')
print(neg[neg > 0] if neg.sum() else 'none found')


In [ ]:
# 4.5 Fix the 28 rows with negative residential asset values
# A negative asset doesn't make sense — most likely a data entry error.
# Clipping to 0 (= applicant has no residential asset) is the safest fix here.
before = int((df[asset_cols] < 0).sum().sum())
df[asset_cols] = df[asset_cols].clip(lower=0)
print(f'Fixed {before} negative asset values → set to 0')
print('Remaining negatives:', int((df[asset_cols] < 0).sum().sum()))


**What I found in the data:**

The dataset is pretty clean honestly — no missing values, no duplicate loan IDs. The only real issue was 28 rows with a negative `residential_assets_value`, which I clipped to 0. I also stripped whitespace from column names and string values — caught a few hidden spaces that would've silently broken my groupby calls later.

| Check | Result |
|---|---|
| Whitespace in names/values | yes, found and removed |
| Missing values | 0 — clean dataset |
| Duplicate loan IDs | 0 |
| CIBIL range | 300–900 ✓ |
| Negative asset values | 28 rows → fixed to 0 |
| Approval labels | only 'Approved' / 'Rejected' ✓ |

All 4,269 rows are usable.


## 5. Feature Engineering

The raw columns describe the applicant but I wanted to add some derived features that capture the *ratios* a lender actually thinks about — how big is this loan compared to what the person earns, and compared to what they own?


In [ ]:
# Sum up all asset types into total_assets
df['total_assets'] = (df['residential_assets_value']
                      + df['commercial_assets_value']
                      + df['luxury_assets_value']
                      + df['bank_asset_value'])

# Loan-to-income: how many years of income the loan represents
df['loan_to_income'] = df['loan_amount'] / df['income_annum']

# Loan-to-assets: what fraction of total collateral the loan asks for
df['loan_to_assets'] = df['loan_amount'] / df['total_assets']

# CIBIL bands — standard buckets used by Indian lenders
df['cibil_band'] = pd.cut(df['cibil_score'],
                          bins=[299, 550, 650, 750, 901],
                          labels=['300-550', '551-650', '651-750', '751-900'])

df[['cibil_score', 'income_annum', 'loan_amount', 'total_assets',
    'loan_to_income', 'loan_to_assets', 'cibil_band']].head()


## 6. Level 1 — KPIs: What's happening in this portfolio?

Starting with the top-line numbers. My first instinct was that income would strongly separate approved from rejected applicants — let me check if that's true.


**My first hypothesis: higher income → more approvals.** Let me check the income distributions before building the full KPI table.

In [ ]:
# Quick check — does income separate approved from rejected?
# I expected a big gap here
approved = df[df['loan_status'] == 'Approved']
rejected = df[df['loan_status'] == 'Rejected']

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist([approved['income_annum']/1e5, rejected['income_annum']/1e5],
        bins=30, label=['Approved', 'Rejected'],
        color=[PRIMARY, RED], alpha=0.65, edgecolor='white')
ax.set_title('My initial hypothesis: income drives approvals')
ax.set_xlabel('Annual income (INR lakh)')
ax.set_ylabel('Applications')
ax.legend()
plt.tight_layout()
plt.show()

print('Avg income - Approved :', round(approved['income_annum'].mean()/1e5, 1), 'L')
print('Avg income - Rejected :', round(rejected['income_annum'].mean()/1e5, 1), 'L')


The distributions overlap almost completely — the averages are nearly identical (≈50 L vs ≈51 L). So income by itself doesn't explain decisions. Interesting. Moving on to the full KPI table.

In [ ]:
kpi = pd.DataFrame({
    'Metric': [
        'Total applications',
        'Approved applications',
        'Rejected applications',
        'Overall approval rate (%)',
        'Total loan book of approved loans (INR crore)',
        'Average income — approved (INR lakh)',
        'Average income — rejected (INR lakh)',
        'Average CIBIL — approved',
        'Average CIBIL — rejected',
        'Average loan size — approved (INR lakh)',
    ],
    'Value': [
        len(df),
        len(approved),
        len(rejected),
        round((df['loan_status'] == 'Approved').mean() * 100, 1),
        round(approved['loan_amount'].sum() / 1e7, 1),
        round(approved['income_annum'].mean() / 1e5, 1),
        round(rejected['income_annum'].mean() / 1e5, 1),
        round(approved['cibil_score'].mean(), 1),
        round(rejected['cibil_score'].mean(), 1),
        round(approved['loan_amount'].mean() / 1e5, 1),
    ],
})
kpi


In [ ]:
# KPI card dashboard
fig, axes = plt.subplots(2, 3, figsize=(11.5, 6))
cards = [
    ('Applications', f'{len(df):,}', '4,269 applications reviewed'),
    ('Approval rate', f"{kpi.loc[3,'Value']}%", 'Approved vs Rejected'),
    ('Loan book (approved)', f"Rs {kpi.loc[4,'Value']} cr", 'Total sanctioned value'),
    ('Avg CIBIL approved', f'{kpi.loc[7,"Value"]}', 'Credit health of approved'),
    ('Avg CIBIL rejected', f'{kpi.loc[8,"Value"]}', 'Credit risk segment'),
    ('Avg income approved', f"Rs {kpi.loc[5,'Value']} L", 'Per applicant, per year'),
]
for ax, (title, big, sub) in zip(axes.flat, cards):
    ax.axis('off')
    ax.add_patch(plt.Rectangle((0.04, 0.10), 0.92, 0.80,
                               facecolor='#F4F7FB', edgecolor=PRIMARY, linewidth=1.2,
                               transform=ax.transAxes))
    ax.text(0.5, 0.68, big, ha='center', va='center', fontsize=22,
            fontweight='bold', color=PRIMARY, transform=ax.transAxes)
    ax.text(0.5, 0.34, title, ha='center', va='center', fontsize=12.5,
            fontweight='bold', color='#1F2A37', transform=ax.transAxes)
    ax.text(0.5, 0.18, sub, ha='center', va='center', fontsize=9,
            color='#525252', transform=ax.transAxes)
fig.suptitle('Loan Portfolio — Executive KPI Overview', fontsize=15, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.94])
fig.savefig('outputs/fig0_kpi_dashboard.png', dpi=150)
plt.show()


This is where my income hypothesis fell apart. Approved and rejected applicants earn almost exactly the same — 50.3 L vs 51.1 L — so the bank clearly isn't filtering on income. But look at the CIBIL gap: **703 vs 429**. That's a massive difference. That told me exactly where to look next.


## 7. Level 2 & 3 — Trends & Drivers: Why do approvals happen?

The KPIs showed *what* — now I want to know *why*. Starting with CIBIL since the averages already pointed there.


### 7.1 CIBIL score — the main story

In [ ]:
appr_by_band = (df.groupby('cibil_band', observed=True)['loan_status']
                     .apply(lambda s: (s == 'Approved').mean() * 100).round(1))

fig, ax = plt.subplots(figsize=(8.5, 4.8))
colors = [RED, ACCENT, PRIMARY, GREEN]
ax.bar(appr_by_band.index.astype(str), appr_by_band.values, color=colors, edgecolor='white')
for i, v in enumerate(appr_by_band.values):
    ax.text(i, v + 1.5, f'{v}%', ha='center', fontsize=11, fontweight='bold')
ax.set_title('Approval Rate by CIBIL Score Band')
ax.set_xlabel('CIBIL score band')
ax.set_ylabel('Approval rate (%)')
ax.set_ylim(0, 110)
plt.tight_layout()
fig.savefig('outputs/fig1_cibil_band_approval.png', dpi=150)
plt.show()


This is the most important chart in the whole project. Below 550 CIBIL: approved 10.6% of the time. Above 550: approved 99%+ of the time. That's basically a binary cutoff, not a continuous score. The bank has what amounts to a hard floor at ~550 — once you're above it, you're almost certainly getting the loan; below it you're almost certainly not.

What's interesting is that I never saw this stated anywhere in the dataset — it just came out of the analysis.


### 7.2 CIBIL distribution — approved vs rejected

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
axes[0].hist([approved['cibil_score'], rejected['cibil_score']],
             bins=40, label=['Approved', 'Rejected'],
             color=[PRIMARY, RED], alpha=0.75, edgecolor='white')
axes[0].set_title('CIBIL Score Distribution by Decision')
axes[0].set_xlabel('CIBIL score')
axes[0].set_ylabel('Applications')
axes[0].legend()

bp = axes[1].boxplot([approved['cibil_score'], rejected['cibil_score']],
                     labels=['Approved', 'Rejected'], patch_artist=True,
                     medianprops=dict(color='black', linewidth=1.5))
for patch, color in zip(bp['boxes'], [PRIMARY, RED]):
    patch.set_facecolor(color); patch.set_alpha(0.5)
axes[1].set_title('CIBIL Score — Box Plot by Decision')
axes[1].set_ylabel('CIBIL score')
plt.tight_layout()
fig.savefig('outputs/fig2_cibil_distribution.png', dpi=150)
plt.show()


### 7.3 What about education, employment, and dependents?

I expected self-employment or having more dependents to hurt approval odds. Let me check.

In [ ]:
def approval_rate(col):
    return (df.groupby(col, observed=True)['loan_status']
            .apply(lambda s: (s == 'Approved').mean() * 100).round(1))

fig, axes = plt.subplots(1, 3, figsize=(12.5, 4.3))
for ax, col in zip(axes, ['education', 'self_employed', 'no_of_dependents']):
    rates = approval_rate(col)
    ax.bar(rates.index.astype(str), rates.values,
           color=[PRIMARY if v >= 62 else ACCENT for v in rates.values],
           edgecolor='white')
    for i, v in enumerate(rates.values):
        ax.text(i, v + 1, f'{v}%', ha='center', fontsize=9.5)
    ax.set_title(col.replace('_', ' ').title())
    ax.set_ylabel('Approval rate (%)')
    ax.set_ylim(0, 100)
plt.suptitle('Approval Rate by Demographic Factors', fontsize=14, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.94])
fig.savefig('outputs/fig3_neutral_factors.png', dpi=150)
plt.show()


These charts surprised me. The approval rate is basically flat across all three — graduate vs not graduate (62.5% vs 62.0%), self-employed vs salaried (62.2% vs 62.2%), even across different numbers of dependents (60–64%). None of these factors move the needle at all.

From a fairness perspective that's actually a good sign — the bank isn't discriminating on who the person is, only on their credit profile. But it also means these columns add almost no predictive power to a model.


### 7.4 Loan size relative to income and assets

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.6))

li = (df.assign(b=pd.cut(df['loan_to_income'], [0, 0.5, 1, 2, 1000],
                        labels=['<=0.5', '0.5-1', '1-2', '>2']))
      .groupby('b', observed=True)['loan_status']
      .apply(lambda s: (s == 'Approved').mean() * 100).round(1))
axes[0].bar(li.index.astype(str), li.values, color=PRIMARY, edgecolor='white')
for i, v in enumerate(li.values):
    if not pd.isna(v):
        axes[0].text(i, v + 1, f'{v}%', ha='center', fontsize=9.5)
axes[0].set_title('Approval Rate by Loan-to-Income Ratio')
axes[0].set_ylabel('Approval rate (%)')
axes[0].set_ylim(0, 100)

la = (df.assign(b=pd.cut(df['loan_to_assets'], [0, 0.5, 1, 3, 1000],
                        labels=['<=0.5', '0.5-1', '1-3', '>3']))
      .groupby('b', observed=True)['loan_status']
      .apply(lambda s: (s == 'Approved').mean() * 100).round(1))
axes[1].bar(la.index.astype(str), la.values,
            color=[GREEN, GREEN, RED, GRAY], edgecolor='white')
for i, v in enumerate(la.values):
    if not pd.isna(v):
        axes[1].text(i, v + 1, f'{v}%', ha='center', fontsize=9.5)
axes[1].set_title('Approval Rate by Loan-to-Assets Ratio')
axes[1].set_ylabel('Approval rate (%)')
axes[1].set_ylim(0, 100)
plt.tight_layout()
fig.savefig('outputs/fig4_loan_ratios.png', dpi=150)
plt.show()


The loan-to-assets ratio is the clearest hard rule I found: **if the loan amount exceeds the applicant's total assets, the approval rate is literally 0%.** Not close to zero — zero. The bank never sanctions a loan bigger than the collateral backing it. That's a firm rule.

The loan-to-income ratio is softer — approval dips in the 1–2× band (53.3%) but not as dramatically. High earners asking for large loans (>2×) still get approved at 62.5%, probably because their income already tells a strong story.


## 8. Level 4 — Risks & Opportunities

Based on the EDA so far, here's how I'd frame the risk and opportunity picture:

| | What I found |
|---|---|
| ⚠ **Risk 1** | The CIBIL cutoff at ~550 excludes about 59% of applicants. That's the biggest single rejection driver. |
| ⚠ **Risk 2** | The collateral ceiling (loan > assets = 0% approval) blocks asset-light applicants — young professionals with good incomes but limited property. |
| ⚠ **Risk 3** | Because CIBIL explains almost everything, the whole process is essentially one number. If that score is gamed or miscalibrated, the entire decision flips. |
| ✦ **Opportunity 1** | Demographics don't matter at all for approvals — so the bank can market freely to self-employed people, non-graduates, large families etc. without any extra risk. |
| ✦ **Opportunity 2** | ~245 applicants in the sub-550 band *were* approved — meaning low-CIBIL lending is possible with the right profile. A small pilot could explore this safely. |
| ✦ **Opportunity 3** | A transparent model (next section) means the bank can tell rejected applicants *exactly* what held them back, which is better UX and probably leads to faster re-applications. |


## 9. Prediction Model

I went with Logistic Regression as my base model because I wanted to be able to explain the output — in banking you need to be able to say "this was rejected because of X", not just produce a probability. Random Forest will almost certainly be more accurate, but I'll add it at the end as a cross-check rather than the main model.


In [ ]:
# Encode categorical columns manually — easier to inspect than pd.get_dummies for a small feature set
X = df[['cibil_score', 'income_annum', 'loan_amount', 'loan_term',
        'no_of_dependents', 'total_assets', 'loan_to_income', 'loan_to_assets',
        'education', 'self_employed']].copy()
X['education_Graduate'] = (X['education'] == 'Graduate').astype(int)
X['self_employed_Yes'] = (X['self_employed'] == 'Yes').astype(int)
X = X.drop(columns=['education', 'self_employed'])

y = (df['loan_status'] == 'Approved').astype(int)

# Stratified split — I forgot stratify=y the first time and the approval ratio
# in my test set came out slightly different (64% instead of 62%).
# Adding stratify=y keeps the same 62/38 split in both train and test.
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)  # no stratify — wrong
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

model = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=2000, random_state=42)),
])
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print('Train accuracy:', round(model.score(X_train, y_train), 4))
print('Test accuracy :', round(accuracy_score(y_test, y_pred), 4))


In [ ]:
# Full report on the held-out 20% test set
print(classification_report(y_test, y_pred,
                            target_names=['Rejected (0)', 'Approved (1)']))
print('ROC-AUC:', round(roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]), 4))


In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5.6, 4.6))
ax.imshow(cm, cmap='Blues')
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(['Rejected', 'Approved'])
ax.set_yticklabels(['Rejected', 'Approved'])
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                fontsize=15, fontweight='bold',
                color='white' if cm[i, j] > cm.max()/2 else 'black')
ax.set_title('Confusion Matrix — Held-out Test Set')
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.tight_layout()
fig.savefig('outputs/fig5_confusion_matrix.png', dpi=150)
plt.show()


In [ ]:
# Feature importance from logistic coefficients
# Positive = pushes toward Approval, Negative = pushes toward Rejection
coefs = model.named_steps['clf'].coef_[0]
feat_imp = pd.Series(coefs, index=X.columns).sort_values()

fig, ax = plt.subplots(figsize=(9, 4.8))
colors = [GREEN if v >= 0 else RED for v in feat_imp.values]
ax.barh(feat_imp.index, feat_imp.values, color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('What Drives Approval — Logistic Regression Coefficients (scaled)')
ax.set_xlabel('Effect on approval probability')
plt.tight_layout()
fig.savefig('outputs/fig6_feature_effects.png', dpi=150)
plt.show()

# Random Forest as a cross-check (more accurate but less explainable)
rf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print('Random Forest test accuracy:', round(rf.score(X_test, y_test), 4))
print('Top-3 RF feature importances:', rf_imp.head(3).round(3).to_dict())


The logistic model hits **91.1% accuracy and ROC-AUC 0.973** on the held-out test set, which is solid. The Random Forest cross-check gets to 99.9%, which confirms the patterns are real — the decision rules are clean enough that a tree model can basically memorize them.

The coefficient chart confirms everything the EDA showed: CIBIL score is by far the biggest driver (and carries 82% of the Random Forest's feature importance too). `loan_to_assets` and `total_assets` push toward rejection. Education and employment type are basically flat — matching the 62% rates across groups we saw earlier.

I'm keeping logistic regression as the main model specifically because it's explainable. For a bank that needs to tell a customer "here's why you were rejected", a coefficient you can translate into a reason is worth more than extra accuracy points.


## 10. Level 5 — Recommendations

Based on everything above, here's what I'd actually suggest to the bank:

The most obvious thing is to **be transparent about the CIBIL floor**. Right now applicants below ~550 get rejected without knowing why. If you tell them "your CIBIL score is 490 — you need to get it above 550, here's how", you turn a frustrated rejected customer into a potential customer 6 months later. That's just better business.

The **collateral rule** (loan can't exceed total assets) is strict but probably correct. What the bank could do better is explain it upfront — applicants who are blocked here can often fix it by adding a co-borrower or guarantor. A lot of people don't know that.

The demographic analysis was genuinely useful here — since education, employment status, and number of dependents don't affect approval odds at all, the bank can **run segment-specific marketing campaigns** to self-employed people, large families etc. without taking on extra risk. They're being filtered on CIBIL, not on who they are.

The **~245 approved sub-550 applicants** are worth studying more. There's clearly something about their profiles that worked — smaller loan amounts, larger assets, better income ratios. A pilot programme with tighter conditions (smaller tickets, co-borrowers) could safely test whether the bank can serve the thin-credit segment, which is currently 59% of all applicants being turned away.

Finally, the model itself could be deployed as a **pre-screening layer** in the application portal — not to replace officers, but to give applicants an instant preliminary read and flag the top 2-3 reasons for a likely rejection. That improves the experience for both the applicant and the officer.


## 11. Conclusion

Coming in I expected income to be the main factor. It isn't — approved and rejected applicants earn almost the same. The real story is almost entirely in the CIBIL score, with a hard-looking floor around 550 and a secondary collateral rule that blocks loans above total asset value.

What I found most interesting is how *clean* the decision pattern is. The Random Forest can predict it at 99.9% accuracy, which means the bank is essentially applying a small number of consistent rules rather than making nuanced judgement calls. That's good for consistency but also means the decisions are explainable — which is exactly what the logistic regression model can deliver.

The 59% of applicants in the low-CIBIL band are both the biggest exclusion and potentially the biggest opportunity. The data shows it's not impossible to approve them (245 were), just risky without the right conditions. That feels like the most interesting business question to explore next.

---

**What I'd look into with more time:**
- Segment the ~245 approved sub-550 applicants to see what made them different — lower loan amounts? higher assets?
- Try adding a monotone constraint to the CIBIL coefficient so the model can't predict approval at lower scores (to force the rule to be explicit)
- Build a simple interactive tool where you enter an applicant's details and get back a decision + top 3 reasons in plain language

---
*AICTE | IBM SkillsBuild Internship Program 2026 — Data Analytics with AI (Sponsored by BharatCares).*
*Intern: Gupta Vaishnavi Sureshbhai | Tools: Python, Pandas, NumPy, Matplotlib, Scikit-learn, Jupyter Notebook.*
